In [1]:
pip install chromadb sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

# ── 1. Load the embedding model ──────────────────────────────────────────────
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# ── 2. Set up ChromaDB ────────────────────────────────────────────────────────
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="cricket_docs")

# ── 3. Chunking function ──────────────────────────────────────────────────────
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Splits text into overlapping chunks of words.
    - chunk_size: how many words per chunk
    - overlap: how many words to repeat between chunks (keeps context)
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # step forward, but overlap a bit

    return chunks

# ── 4. Load, chunk, embed, and store all documents ───────────────────────────
docs_folder = "./cricket_docs"
txt_files = [f for f in os.listdir(docs_folder) if f.endswith(".txt")]

print(f"Found {len(txt_files)} documents to process...\n")

total_chunks = 0

for filename in txt_files:
    filepath = os.path.join(docs_folder, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    # Skip empty files
    if not text.strip():
        print(f"⚠️  Skipping empty file: {filename}")
        continue

    # Split into chunks
    chunks = chunk_text(text)

    # Embed all chunks at once (faster than one by one)
    embeddings = model.encode(chunks).tolist()

    # Store in ChromaDB
    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=[f"{filename}_chunk_{i}" for i in range(len(chunks))],
        metadatas=[{"source": filename} for _ in chunks]
    )

    total_chunks += len(chunks)
    print(f"✅ {filename}: {len(chunks)} chunks stored")

print(f"\n🎉 Done! {total_chunks} total chunks stored in ChromaDB.")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4889.57it/s]


Found 43 documents to process...

✅ Swing_bowling.txt: 4 chunks stored
✅ Don_Bradman.txt: 28 chunks stored
✅ Cricket.txt: 22 chunks stored
✅ Brian_Lara.txt: 8 chunks stored
✅ Sachin_Tendulkar.txt: 27 chunks stored
✅ Indian_Premier_League.txt: 13 chunks stored
✅ Indian_cricket_team.txt: 18 chunks stored
✅ England_cricket_team.txt: 19 chunks stored
✅ Virat_Kohli.txt: 17 chunks stored
✅ Ricky_Ponting.txt: 41 chunks stored
✅ Test_cricket.txt: 21 chunks stored
✅ Laws_of_Cricket.txt: 14 chunks stored
✅ Champions_Trophy.txt: 8 chunks stored
✅ Century_(cricket).txt: 3 chunks stored
✅ Bowling_(cricket).txt: 8 chunks stored
✅ Wide_(cricket).txt: 2 chunks stored
✅ Stumping.txt: 2 chunks stored
✅ Shane_Warne.txt: 18 chunks stored
✅ Dismissal_(cricket).txt: 7 chunks stored
✅ ICC_World_Twenty20.txt: 6 chunks stored
✅ LBW.txt: 1 chunks stored
✅ MS_Dhoni.txt: 11 chunks stored
✅ West_Indies_cricket_team.txt: 10 chunks stored
✅ Imran_Khan.txt: 24 chunks stored
✅ Pakistan_cricket_team.txt: 16 chunks stor

In [ ]:
           
   # what files are actually there

/Users/ajayrao/Desktop/Cricket_Rag_Chatbot/Rag_chatbot
['embed_and_store.ipynb', 'cricket_rag_scraper.ipynb', 'chroma_db']
